# Related Symbolic Music Corpora

This chapter introduces two publicly available symbolic music datasets that complement the **Turkish Delight Corpus (TDC)** and provide additional resources for phrase-level analysis of Turkish makam music.

The datasets presented in this chapter are:

* **Turkish Makam Symbolic Phrase Segmentation Dataset**
  Resource page: https://compmusic.upf.edu/node/237

* **Turkish Makam Melodic Phrase Dataset**
  Resource page: https://compmusic.upf.edu/node/236

Together, these corpora extend the scope of the TDC by providing expert-curated phrase annotations and complementary symbolic representations that support comparative analyses, phrase segmentation research, symbolic music processing, and future Music Information Retrieval (MIR) and machine learning studies.

The corresponding resource pages provide additional information about each dataset, including their documentation, metadata, and download instructions.


## 1. Environment Preparation

Before accessing the related symbolic music datasets, the required Python libraries are imported and the computational environment is initialized.

The notebook relies on **Pathlib** for platform-independent file management, **Requests** for communicating with the Zenodo REST API and downloading datasets, **ZipFile** for archive extraction, and **Pandas** for organizing and presenting the collected metadata.

Preparing the computational environment in advance ensures that all subsequent analyses are performed within a reproducible and portable workflow.

In [9]:
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import requests

print("Environment ready.")

Environment ready.


## 2. Prepare the Project Directories

To ensure a reproducible project structure, the notebook automatically locates the root directory of the **TDC-Analysis-Book** project and creates dedicated directories for storing the related datasets.

Each dataset is assigned its own local directory under:

```text
data/raw/related
```

This organization preserves the original dataset archives while providing a scalable structure for incorporating additional symbolic music datasets in future extensions of the notebook.

In [10]:
current_directory = Path.cwd()
project_root = current_directory

while (
    project_root.name != "TDC-Analysis-Book"
    and project_root.parent != project_root
):
    project_root = project_root.parent

if project_root.name != "TDC-Analysis-Book":
    raise FileNotFoundError(
        "The TDC-Analysis-Book project directory could not be located."
    )

related_data_dir = project_root / "data" / "raw" / "related"

symbolic_phrase_dir = related_data_dir / "symbolic_phrase"
melodic_phrase_dir = related_data_dir / "melodic_phrase"

symbolic_phrase_dir.mkdir(parents=True, exist_ok=True)
melodic_phrase_dir.mkdir(parents=True, exist_ok=True)

print("Project directories initialized.")

Project directories initialized.


## 3. Register the Related Datasets

The related symbolic music datasets are registered using a structured Python dictionary.

For each dataset, the notebook stores its official title, the corresponding Music Technology Group (MTG) resource page, the Zenodo record identifier, and the local directory used for storing the downloaded archive.

Using the Zenodo record identifier instead of manually specifying download filenames improves the robustness and long-term maintainability of the notebook.

In [11]:
related_datasets = {
    "symbolic_phrase": {
        "title": "Turkish Makam Symbolic Phrase Segmentation Dataset",
        "resource_url": "https://compmusic.upf.edu/node/237",
        "record_id": 168208,
        "local_directory": symbolic_phrase_dir,
    },
    "melodic_phrase": {
        "title": "Turkish Makam Melodic Phrase Dataset",
        "resource_url": "https://compmusic.upf.edu/node/236",
        "record_id": 1283344,
        "local_directory": melodic_phrase_dir,
    },
}

print(f"Registered datasets: {len(related_datasets)}")

Registered datasets: 2


## 4. Retrieve and Download the Related Datasets

The notebook retrieves the official file metadata from the Zenodo REST API and downloads the corresponding dataset archives into their designated local directories.

For each registered dataset, the workflow identifies the available ZIP archive, obtains its official download URL, checks whether the archive already exists locally, and downloads only missing files. Combining metadata retrieval and downloading within a single workflow prevents errors caused by missing intermediate variables or out-of-order notebook execution.

For each dataset, the notebook reports the archive filename, download status, and local file size.

In [12]:
download_results = []

for dataset in related_datasets.values():

    dataset_title = dataset["title"]
    record_id = dataset["record_id"]
    local_directory = dataset["local_directory"]

    local_directory.mkdir(parents=True, exist_ok=True)

    try:
        # Retrieve the official Zenodo record metadata
        api_url = f"https://zenodo.org/api/records/{record_id}"

        metadata_response = requests.get(
            api_url,
            timeout=60,
        )
        metadata_response.raise_for_status()

        metadata = metadata_response.json()

        # Select the ZIP archive from the files registered in Zenodo
        zip_files = [
            file_info
            for file_info in metadata.get("files", [])
            if file_info["key"].lower().endswith(".zip")
        ]

        if not zip_files:
            raise FileNotFoundError(
                f"No ZIP archive was found in Zenodo record {record_id}."
            )

        file_info = zip_files[0]

        # Remove any directory prefix contained in the Zenodo file key
        archive_name = Path(file_info["key"]).name
        download_url = file_info["links"]["self"]

        archive_path = local_directory / archive_name

        # Reuse an existing local archive when available
        if archive_path.exists():
            status = "Already downloaded"

        else:
            download_response = requests.get(
                download_url,
                stream=True,
                timeout=120,
                allow_redirects=True,
            )
            download_response.raise_for_status()

            with archive_path.open("wb") as archive_file:
                for chunk in download_response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        archive_file.write(chunk)

            status = "Downloaded"

        # Store the retrieved information for later notebook sections
        dataset["archive_name"] = archive_name
        dataset["archive_path"] = archive_path
        dataset["download_url"] = download_url

        download_results.append({
            "Dataset": dataset_title,
            "Archive": archive_name,
            "Status": status,
            "Size (MB)": archive_path.stat().st_size / (1024 ** 2),
        })

    except (
        requests.RequestException,
        FileNotFoundError,
        KeyError,
        ValueError,
    ) as error:

        download_results.append({
            "Dataset": dataset_title,
            "Archive": None,
            "Status": "Download failed",
            "Size (MB)": None,
        })

        print(f"Download failed for {dataset_title}: {error}")

download_df = pd.DataFrame(download_results)

download_df.index = range(1, len(download_df) + 1)
download_df.index.name = ""

download_df.style \
    .set_properties(**{"text-align": "left"}) \
    .format({"Size (MB)": "{:.2f}"}, na_rep="-") \
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                ],
            },
        ],
        overwrite=False,
    )

,Dataset,Archive,Status,Size (MB)
,,,,
1,Turkish Makam Symbolic Phrase Segmentation Dataset,otmm_symbolic_phrase_dataset-v1.0.zip,Already downloaded,4.01
2,Turkish Makam Melodic Phrase Dataset,turkish_makam_melodic_phrase_1.0.zip,Already downloaded,21.54


## Download Summary

The related symbolic music datasets were successfully downloaded from their official Zenodo repositories.

For each dataset, the notebook reports the archive filename, download status, and archive size. If an archive already exists in its designated local directory, the download is skipped to avoid unnecessary network transfers while preserving the existing local copy. This mechanism supports efficient, repeatable, and reproducible execution of the notebook.

The downloaded archive files are stored in their corresponding subdirectories under:

```text
data/raw/related
```

These archive files represent the original dataset distributions and will be extracted in the following section for structural exploration and subsequent analyses.

## Extracting the Downloaded Dataset Archives

The downloaded dataset archives are extracted into their corresponding local directories while preserving the original directory structure.

Before extraction, the workflow checks whether the dataset has already been extracted. Existing extracted directories are preserved to avoid unnecessary operations, ensuring efficient and reproducible execution of the notebook.

After extraction, the datasets become available for structural exploration, metadata inspection, and subsequent analyses presented in the following sections.

In [13]:
from zipfile import ZipFile
import pandas as pd

extraction_results = []

for dataset in related_datasets.values():

    zip_files = list(dataset["local_directory"].glob("*.zip"))

    if not zip_files:
        extraction_results.append({
            "Dataset": dataset["title"],
            "Status": "Archive not found",
        })
        continue

    archive_path = zip_files[0]

    extract_directory = (
        dataset["local_directory"] / "extracted"
    )

    if extract_directory.exists():

        extraction_results.append({
            "Dataset": dataset["title"],
            "Status": "Already extracted",
        })

        continue

    extract_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    with ZipFile(archive_path, "r") as zip_file:
        zip_file.extractall(extract_directory)

    extraction_results.append({
        "Dataset": dataset["title"],
        "Status": "Extracted",
    })

extraction_df = pd.DataFrame(extraction_results)

extraction_df.index = range(
    1,
    len(extraction_df) + 1,
)
extraction_df.index.name = ""

extraction_df.style \
    .set_properties(**{"text-align": "left"}) \
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                ],
            },
        ],
        overwrite=False,
    )

,Dataset,Status
,,
1,Turkish Makam Symbolic Phrase Segmentation Dataset,Already extracted
2,Turkish Makam Melodic Phrase Dataset,Already extracted


### Interpretation

Both related symbolic music datasets were successfully extracted into their designated local directories while preserving their original directory structures.

The extracted contents are now directly accessible for further exploration and analysis. Preserving the original organization of each dataset ensures that symbolic music files, annotations, metadata, and accompanying documentation remain consistent with the official dataset distributions.

This extraction step establishes a reproducible foundation for the next stage of the notebook, where the internal directory structure and file organization of each dataset are systematically examined.

## Inspecting the Extracted Directory Structure

After extracting the dataset archives, the top-level directory structure of each dataset is inspected to verify that the extraction process was completed successfully and that the original organization has been preserved.

The following output lists the top-level directories and files contained in each extracted dataset. This preliminary inspection provides an overview of the dataset organization and establishes a foundation for the subsequent structural and statistical analyses presented in this chapter.

In [14]:
from pathlib import Path

for dataset in related_datasets.values():

    print("=" * 70)
    print(f"Dataset: {dataset['title']}")
    print("=" * 70)

    extract_directory = (
        dataset["local_directory"] /
        "extracted"
    )

    items = sorted(extract_directory.iterdir())

    print(f"Top-level items: {len(items)}\n")

    for item in items:

        icon = "📁" if item.is_dir() else "📄"
        print(f"{icon} {item.name}")

    print()

Dataset: Turkish Makam Symbolic Phrase Segmentation Dataset
Top-level items: 1

📁 MTG-otmm_symbolic_phrase_dataset-2f6cca9

Dataset: Turkish Makam Melodic Phrase Dataset
Top-level items: 1

📁 turkish_makam_melodic_phrase_1.0



## Inspecting the Internal Dataset Structure

After verifying the top-level organization of the extracted archives, the internal directory structure of each dataset is inspected recursively.

Each archive contains a single dataset-specific root directory that organizes symbolic scores, annotations, metadata, documentation, and other supporting resources into a hierarchical structure. Exploring this organization provides a better understanding of the dataset contents before performing quantitative analyses.

To maintain the readability of the notebook, only the first levels of the directory hierarchy are displayed rather than listing every individual file contained in the datasets.

In [15]:
from pathlib import Path

def display_directory_tree(
    root_directory: Path,
    max_depth: int = 2,
) -> None:
    """
    Display a compact directory tree up to a specified depth.
    """
    root_directory = Path(root_directory)

    for path in sorted(root_directory.rglob("*")):

        relative_path = path.relative_to(root_directory)
        depth = len(relative_path.parts)

        if depth > max_depth:
            continue

        indentation = "    " * (depth - 1)
        icon = "📁" if path.is_dir() else "📄"

        print(f"{indentation}{icon} {relative_path.name}")


for dataset in related_datasets.values():

    extract_directory = (
        dataset["local_directory"] /
        "extracted"
    )

    root_items = sorted(extract_directory.iterdir())

    if len(root_items) != 1 or not root_items[0].is_dir():

        print(f"Unexpected directory structure: {dataset['title']}")
        continue

    dataset_root = root_items[0]

    print("=" * 70)
    print(f"Dataset: {dataset['title']}")
    print(f"Root directory: {dataset_root.name}")
    print("=" * 70)

    display_directory_tree(
        dataset_root,
        max_depth=2,
    )

    print()

Dataset: Turkish Makam Symbolic Phrase Segmentation Dataset
Root directory: MTG-otmm_symbolic_phrase_dataset-2f6cca9
📄 .gitignore
📁 annotations
    📁 expert1
    📁 expert2
    📁 expert3
    📄 symbtr_mbid.csv
📁 extras
    📄 utf8_unixline_converter.ipynb
📄 karaosmanoglu2014symbtrPhrase_fma.pdf
📄 LICENSE
📄 README.md

Dataset: Turkish Makam Melodic Phrase Dataset
Root directory: turkish_makam_melodic_phrase_1.0
📁 112E162
    📄 .DS_Store
    📄 5eser_pdf.rar
    📄 5eser_symbtr.rar
    📄 matlabToolsForAutoSeg.zip
    📄 mid_500_1.rar
    📄 mid_500_2.rar
    📄 uzman1_symbtr_txt.zip
    📄 uzman2_symbtr_txt.zip
    📄 uzman3_symbtr_txt.zip
📁 __MACOSX
    📄 ._112E162
    📁 112E162



The directory trees reveal clear differences in the internal organization and resource composition of the two datasets.

The **Turkish Makam Symbolic Phrase Segmentation Dataset** is organized into dedicated directories for annotations, expert-generated resources, and supplementary materials. In addition to symbolic music files, the dataset includes metadata, documentation, licensing information, and supporting resources, indicating a well-structured collection designed for phrase segmentation research and reproducible computational analyses.

In contrast, the **Turkish Makam Melodic Phrase Dataset** has a simpler directory hierarchy centered on melodic phrase resources. The archive primarily contains compressed data files together with supplementary materials generated during the original dataset distribution. The presence of directories such as `__MACOSX` and compressed archives (`.zip` and `.rar`) indicates that the dataset has been preserved in its original distribution format.

Overall, the directory inspection demonstrates that the two datasets differ substantially in their internal organization. The symbolic phrase segmentation dataset provides a richer and more modular structure with annotations and documentation, whereas the melodic phrase dataset offers a more compact archive focused primarily on melodic phrase resources. These observations provide useful context for the structural statistics and comparative analyses presented in the following sections.

## Analyzing File Types

Following the structural inspection of the extracted datasets, the distribution of file types is examined to better understand the composition of each resource.

Each dataset consists of multiple file formats serving different purposes, including symbolic music representations, annotation files, metadata, documentation, compressed archives, and supplementary resources. By grouping files according to their extensions, the workflow provides a concise overview of the available data formats and highlights the primary resources contained in each dataset.

This analysis facilitates the identification of files that are directly relevant to subsequent computational analyses while distinguishing them from supporting documentation and auxiliary resources.

In [16]:
from collections import Counter
from pathlib import Path

filetype_results = []

for dataset in related_datasets.values():

    extract_directory = (
        dataset["local_directory"] /
        "extracted"
    )

    dataset_root = next(extract_directory.iterdir())

    extension_counter = Counter()

    for file_path in dataset_root.rglob("*"):

        if not file_path.is_file():
            continue

        extension = (
            file_path.suffix.lower()
            if file_path.suffix
            else "[no extension]"
        )

        extension_counter[extension] += 1

    for extension, count in sorted(extension_counter.items()):

        filetype_results.append(
            {
                "Dataset": dataset["title"],
                "Extension": extension,
                "Files": count,
            }
        )

filetype_df = pd.DataFrame(filetype_results)

filetype_df.style \
    .hide(axis="index") \
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                ],
            }
        ],
        overwrite=False,
    ) \
    .set_properties(
        subset=["Dataset", "Extension"],
        **{
            "text-align": "left",
        }
    ) \
    .set_properties(
        subset=["Files"],
        **{
            "text-align": "center",
        }
    )

Dataset,Extension,Files
Turkish Makam Symbolic Phrase Segmentation Dataset,.csv,1
Turkish Makam Symbolic Phrase Segmentation Dataset,.ipynb,1
Turkish Makam Symbolic Phrase Segmentation Dataset,.md,1
Turkish Makam Symbolic Phrase Segmentation Dataset,.pdf,1
Turkish Makam Symbolic Phrase Segmentation Dataset,.txt,888
Turkish Makam Symbolic Phrase Segmentation Dataset,[no extension],2
Turkish Makam Melodic Phrase Dataset,.ds_store,1
Turkish Makam Melodic Phrase Dataset,.rar,8
Turkish Makam Melodic Phrase Dataset,.zip,8
Turkish Makam Melodic Phrase Dataset,[no extension],2


The **Turkish Makam Symbolic Phrase Segmentation Dataset** is composed predominantly of plain-text (`.txt`) files, with **888** text files representing the primary symbolic music data used for phrase segmentation research. In addition, the dataset includes a small number of metadata (`.csv`), documentation (`.md`), and development resources (`.ipynb`), indicating that it is distributed as a complete research package.

A distinguishing characteristic of this dataset is that all melodic phrase boundaries were manually annotated by domain experts. Three independent experts first identified phrase boundaries directly on printed musical scores over a period of approximately three months. These annotations were subsequently transferred into the corresponding SymbTr text files by manually inserting additional annotation records. As a result, the annotated files extend the original SymbTr representation with structural information such as phrase boundaries, measure boundaries, phrase labels, and çeşni/geçki annotations. Consequently, each annotated text file preserves both the original symbolic music representation and the expert-defined structural information required for computational phrase analysis {cite}`karaosmanoglu2014phrases,Bozkurt2014SIU`.

The extracted archive also contains the original research article describing the construction of the dataset. This publication explains the motivation for creating the corpus, the dataset design, the expert annotation methodology, the machine-readable representation, and several potential computational applications. The accompanying studies further demonstrate how these expert annotations have been utilized to develop statistical machine-learning models for automatic melodic phrase segmentation and phrase-based computational analysis of Turkish makam music {cite}`karaosmanoglu2014phrases,Bozkurt2014JNMR,Bozkurt2014SIU`.

Researchers using this dataset are encouraged to consult and appropriately cite the original dataset publication together with the related methodological studies when employing these annotations in computational musicology, Music Information Retrieval (MIR), and machine learning research {cite}`karaosmanoglu2014phrases,Bozkurt2014JNMR`.

The **Turkish Makam Melodic Phrase Dataset** complements the Symbolic Phrase Segmentation Dataset by providing an additional collection of phrase-oriented musical data. Similar to the symbolic phrase dataset, the extracted archive contains multiple file types, including symbolic music files, metadata, documentation, and supplementary research materials. This indicates that the dataset is distributed as a complete research resource rather than as a collection of isolated musical files.

The organization of the extracted files facilitates systematic exploration of the dataset structure and enables researchers to examine the available musical data, accompanying documentation, and supporting resources before performing further computational analyses. Together with the Turkish Makam Symbolic Phrase Segmentation Dataset, it provides an additional dataset for comparative exploration within the TDC Analysis Book.

## Comparative Overview of the Related Datasets

To facilitate a systematic comparison, the structural characteristics of the related symbolic music datasets are summarized in a single table.

The comparison combines the statistics obtained throughout the previous analyses, including the number of directories, total number of files, and the diversity of file types contained in each dataset.

These measurements provide an overview of the internal organization of the datasets rather than their musical content. Comparing these structural characteristics helps researchers understand the composition of each dataset, identify the available resources, and assess their suitability for different computational music research workflows.

### Interpretation

The comparison reveals notable differences in the structural organization of the two datasets. The **Turkish Makam Symbolic Phrase Segmentation Dataset** contains a larger number of files and directories, reflecting its emphasis on expert-annotated symbolic music data together with supporting research materials.

In contrast, the **Turkish Makam Melodic Phrase Dataset** has a more compact directory structure while still containing multiple file types, indicating that it is distributed with accompanying documentation and metadata in addition to the primary musical data.

Although both datasets are organized as complete research resources, their structural characteristics differ in terms of size and organization. This comparison provides a useful overview for researchers before exploring the datasets in greater detail or selecting appropriate resources for subsequent analyses.

## Next Chapter

The chapter, **Related Datasets and Statistical Analysis**, presents publicly available Turkish makam music datasets and compares their statistical characteristics.

The comparative analysis highlights the strengths of existing datasets and provides the motivation for the symbolic music generation workflow presented in the following chapters.